# PyTorch AutoGrad

In [2]:
import torch

## The AutoGrad DAG

In [38]:
def print_graph(fn, indent=0):
    print("  " * indent + str(fn))
    if hasattr(fn, "next_functions"):
        for f, _ in fn.next_functions:
            if f is not None:
                print_graph(f, indent + 1)
    if hasattr(fn, "variable"):
        print("  " * (indent + 1) + str(fn.variable) + f" (grad={fn.variable.grad})")

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
W = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

y = W * x
z = y + b

print("z.grad_fn:")            
print_graph(z.grad_fn)

# Basically:
#               y
#               |
#          MulBackward0
#             /    \
# AccumulateGrad  AccumulateGrad
#      |               |
#      x               W

z.grad_fn:
      tensor(3., requires_grad=True) (grad=None)
      tensor(2., requires_grad=True) (grad=None)
    tensor(4., requires_grad=True) (grad=None)


## The backward() method

In [41]:
x = torch.tensor(2.0, requires_grad=True)
W = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

y = W * x # y = 6
z = y + b # z = 10

z.backward()

print("z.grad_fn:")              
print_graph(z.grad_fn)     # Now grad values are set!

# The gradient of b is simply dz/db = 1
# The gradient of y is simply dz/dy = 1
# The gradient of x (w.r.t. z!) is simply dz/dx = dz/dy * dy/dx = 1 * W = 3
# The gradient of W (w.r.t. z!) is simply dz/dW = dz/dy * dy/dW = 1 * x = 2 

z.grad_fn:
      tensor(3., requires_grad=True) (grad=2.0)
      tensor(2., requires_grad=True) (grad=3.0)
    tensor(4., requires_grad=True) (grad=1.0)


## Multi-Dimensional

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
W = torch.tensor([3.0, 4.0, 5.0], requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

ground_truth = torch.tensor([8.0, 9.0, 12.0], requires_grad=False)

y = W * x # y = 6
z = y + b # z = 10

loss = torch.nn.functional.mse_loss(z, ground_truth)

loss.backward()


# I have for example the gradient of b w.r.t. the loss
print(f"b.grad {b.grad}")
# And of x and W
print(f"x.grad {x.grad}")
print(f"W.grad {W.grad}")
# Notice how only leaf tensors ("AccumulateGrad") have a grad value. 
# All the others only have grad_fn (they have a gradient, it's just not "accumulated" cause we do not needed for optimizer.step())

b.grad 4.6666669845581055
x.grad 18.666667938232422
W.grad tensor([2.6667, 4.0000, 2.6667])


In [55]:
# ----- Data -----
# Input (scalar)
x = torch.tensor(2.0, requires_grad=True)

# Parameters (weights and bias)
W = torch.tensor([3.0, 4.0, 5.0], requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

# Target output
ground_truth = torch.tensor([8.0, 9.0, 12.0])

# ----- Hyperparameters -----
lr = 0.01       # learning rate
epochs = 100    # number of training steps

# ----- Training loop -----
for step in range(epochs):
    # Forward pass
    y = W * x       # linear transformation: elementwise
    z = y + b       # add bias
    loss = torch.nn.functional.mse_loss(z, ground_truth)  # mean squared error

    # Zero gradients (important!)
    # Since we're not using an optimizer, we manually zero gradients
    if W.grad is not None: W.grad.zero_()
    if b.grad is not None: b.grad.zero_()
    if x.grad is not None: x.grad.zero_()  # optional if x is fixed

    # Backward pass
    loss.backward()

    # Gradient descent update (manual)
    with torch.no_grad():  # prevent tracking this update in autograd
        W -= lr * W.grad
        b -= lr * b.grad
        # x -= lr * x.grad  # usually x is input, not trainable

    # Print progress every 10 steps
    if step % 10 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f} | W: {W.tolist()} | b: {b.item():.4f}")

# ----- Final output -----
y_final = W * x + b
print("\nFinal prediction:", y_final)
print("Target ground truth:", ground_truth)
print("Final weights:", W)
print("Final bias:", b)


Step   0 | Loss: 5.6667 | W: [2.9733333587646484, 3.9600000381469727, 4.973333358764648] | b: 3.9533
Step  10 | Loss: 2.2228 | W: [2.7702934741973877, 3.6416985988616943, 4.770293712615967] | b: 3.5911
Step  20 | Loss: 0.8803 | W: [2.6498899459838867, 3.4333314895629883, 4.649890422821045] | b: 3.3666
Step  30 | Loss: 0.3534 | W: [2.579425573348999, 3.295736789703369, 4.579425811767578] | b: 3.2273
Step  40 | Loss: 0.1446 | W: [2.538933753967285, 3.2040138244628906, 4.538933753967285] | b: 3.1409
Step  50 | Loss: 0.0606 | W: [2.516268730163574, 3.1422512531280518, 4.516268730163574] | b: 3.0874
Step  60 | Loss: 0.0263 | W: [2.5040793418884277, 3.100224256515503, 4.504079341888428] | b: 3.0542
Step  70 | Loss: 0.0118 | W: [2.497943878173828, 3.0713179111480713, 4.497944355010986] | b: 3.0336
Step  80 | Loss: 0.0055 | W: [2.495225429534912, 3.051222085952759, 4.495225429534912] | b: 3.0208
Step  90 | Loss: 0.0027 | W: [2.4943687915802, 3.0371031761169434, 4.494368076324463] | b: 3.0129

